# programming-language-identification-100plus

Runnable examples for the ModernBERT programming-language identifier.
Covers 107 languages. Input is truncated to the first 512 characters
(matches the training-time `head` strategy).


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "FrameByFrame/programming-language-identification-100plus"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
).to(DEVICE).eval()

print(f"device={DEVICE}  num_labels={model.config.num_labels}  dtype={model.dtype}")


## Helpers

In [ ]:
@torch.no_grad()
def predict(snippets, top_k=1, max_chars=512):
    """Return the top-k languages + probabilities for each snippet."""
    if isinstance(snippets, str):
        snippets = [snippets]
    trimmed = [s[:max_chars] for s in snippets]
    encoded = tokenizer(
        trimmed, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(DEVICE)
    logits = model(**encoded).logits
    probs = logits.softmax(-1)
    top_probs, top_ids = probs.topk(top_k, dim=-1)
    results = []
    for row_probs, row_ids in zip(top_probs.tolist(), top_ids.tolist()):
        results.append(
            [
                (model.config.id2label[label_id], prob)
                for label_id, prob in zip(row_ids, row_probs)
            ]
        )
    return results


def show(title, snippet, top_k=1):
    preds = predict(snippet, top_k=top_k)[0]
    head = snippet.strip().splitlines()[0][:60]
    print(f"{title:14s}  `{head}`")
    for name, prob in preds:
        print(f"               {name:30s} {prob:.3f}")
    print()

## 1. Single-snippet prediction

In [ ]:
python_snippet = '''
def greet(name: str) -> None:
    print(f"hello, {name}")

for person in ["ada", "alan", "grace"]:
    greet(person)
'''.strip()

show("Python", python_snippet)

## 2. Batch across many languages

In [ ]:
SAMPLES = {
    "Rust": '''
fn main() {
    let names = vec!["ada", "alan", "grace"];
    for n in &names {
        println!("hello, {}", n);
    }
}
'''.strip(),
    "Go": '''
package main

import "fmt"

func main() {
    names := []string{"ada", "alan", "grace"}
    for _, n := range names {
        fmt.Printf("hello, %s\\n", n)
    }
}
'''.strip(),
    "Ruby": '''
["ada", "alan", "grace"].each do |name|
  puts "hello, #{name}"
end
'''.strip(),
    "Elixir": '''
defmodule Greeter do
  def hello(name), do: IO.puts("hello, #{name}")
end

Enum.each(["ada", "alan", "grace"], &Greeter.hello/1)
'''.strip(),
    "Haskell": '''
main :: IO ()
main = mapM_ (\\n -> putStrLn ("hello, " ++ n)) ["ada", "alan", "grace"]
'''.strip(),
    "Kotlin": '''
fun main() {
    listOf("ada", "alan", "grace").forEach { println("hello, $it") }
}
'''.strip(),
    "Mathematica/Wolfram Language": '''
greet[name_String] := Print["hello, " <> name];
greet /@ {"ada", "alan", "grace"};
'''.strip(),
    "ARM Assembly": '''
    .syntax unified
    .thumb
    .global main
main:
    ldr r0, =message
    bl  puts
    mov r0, #0
    bx  lr
message:
    .asciz "hello"
'''.strip(),
    "Julia": '''
for name in ["ada", "alan", "grace"]
    println("hello, $name")
end
'''.strip(),
}

snippets = list(SAMPLES.values())
expected = list(SAMPLES.keys())
predictions = predict(snippets, top_k=1)

correct = 0
for gold, preds in zip(expected, predictions):
    predicted, prob = preds[0]
    mark = "OK " if predicted == gold else "!  "
    print(f"  {mark} gold={gold:32s} pred={predicted:32s} p={prob:.3f}")
    if predicted == gold:
        correct += 1
print(f"\n{correct}/{len(snippets)} correct")

## 3. Top-k with confidence

Useful when a snippet is short or ambiguous — inspect the runner-ups
before committing to a label.

In [ ]:
# Kotlin/Java syntactic overlap — see how far ahead the winner is
jvm_snippet = '''
class Hello {
    fun say(name: String) = println("hello, $name")
}
'''.strip()

show("JVM snippet", jvm_snippet, top_k=5)

## 4. Very short / ambiguous input

Snippets under ~60 characters are often genuinely ambiguous — multiple
languages accept the same syntax. Top-k probabilities will be diffuse.

In [ ]:
show("short", "x = 1", top_k=5)
show("one-liner", "print('hi')", top_k=5)
show("empty-ish", "{}", top_k=5)

## Tips

* Feed at least ~100 characters for reliable results.
* The model was trained and evaluated with the first 512 characters of each
  file. For longer files, that's also what you should pass.
* If you have file extensions available, treat them as a strong prior —
  this classifier is purely content-based and will happily misclassify a
  polyglot hello-world if you ask it to.